# 🎬 ANIME SHORT VIDEO MAKER

Tự động tạo video YouTube Shorts về anime lore/facts (40-60s).
- **Script**: Gemini AI tự sinh kịch bản dựa trên kiến thức anime
- **Ảnh**: Tự tải ảnh nhân vật từ Google Images
- **Giọng đọc**: Gemini 3.1 Flash TTS (không cần GPU!)
- **Video**: FFmpeg ghép ảnh + audio + subtitle

⚡ **Không cần GPU** — chạy trên CPU hoặc bất kỳ runtime nào.

In [ ]:
# @title ⚙️ CELL 1: CÀI ĐẶT (chạy 1 lần, ~1 phút)
import os, subprocess, sys

print('⏳ Đang cài đặt thư viện...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'google-genai', 'requests', 'Pillow', 'python-dotenv'],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

os.system('apt-get install -y -qq ffmpeg 2>/dev/null')

from IPython.display import clear_output
clear_output()

import json, re, time, struct, mimetypes, shutil
from pathlib import Path
from urllib.parse import quote_plus
import requests
from PIL import Image
from google import genai
from google.genai import types

print('✅ TẤT CẢ THƯ VIỆN ĐÃ SẴN SÀNG!')
print('   → Chạy Cell 2 để cấu hình')

In [ ]:
# @title 📝 CELL 2: CẤU HÌNH
# @markdown ---
# @markdown ### Chủ đề video anime
topic = "Rimuru's hidden hero power in That Time I Got Reincarnated as a Slime" # @param {type:'string'}
# @markdown ---
# @markdown ### Gemini API Key (lấy tại aistudio.google.com)
gemini_api_key = '' # @param {type:'string'}
# @markdown ---
# @markdown ### Voice TTS (Achird = calm narration, Charon = deep dramatic)
voice_name = 'Achird' # @param ['Achird', 'Aoede', 'Charon', 'Fenrir', 'Kore', 'Leda', 'Orus', 'Puck', 'Schedar', 'Sulafat', 'Umbriel', 'Zephyr']

assert topic.strip(), '❌ Nhập chủ đề video!'
assert gemini_api_key.strip(), '❌ Nhập Gemini API key!'

slug = re.sub(r'[^a-z0-9]+', '_', topic.lower()).strip('_')[:80]
P = Path(f'/content/{slug}')
P.mkdir(parents=True, exist_ok=True)
(P / 'images').mkdir(exist_ok=True)

print(f'✅ Cấu hình OK!')
print(f'   Project: {slug}')
print(f'   Voice: {voice_name}')
print(f'   → Chạy Cell 3 để tạo video')

In [ ]:
# @title 🚀 CELL 3: TẠO VIDEO (chạy 1 lần, ~3-5 phút)
from IPython.display import Audio, display, HTML
import traceback

print(f"{'='*60}")
print(f'🎬 ANIME SHORT MAKER: {topic}')
print(f"{'='*60}")

# ==========================================
# STEP 1: GENERATE SCRIPT
# ==========================================
print(f'\n📖 [1/5] Sinh kịch bản anime (Gemini AI)...')

PROMPT = '''You are an expert anime content creator who makes viral YouTube Shorts about hidden anime lore, unknown facts, and shocking revelations. Write an engaging narration script about: "{topic}"

## CRITICAL RULES:
- You MUST use ACCURATE anime lore. Do NOT make up facts.
- The script MUST be between 150 and 180 words.
- Write in a flowing, dramatic narration style.
- Use short, punchy sentences. Build tension and curiosity.

## STRUCTURE:
- Opening Hook (~25 words): Start with a shocking statement.
- Body (~120 words): Reveal the lore step by step.
- Closing (~25 words): End with a mind-blowing conclusion.

## PUNCTUATION FOR TTS:
- Periods (.) for natural pauses. Commas (,) for flowing continuation.
- Blank lines between paragraphs = dramatic pause.
- NEVER use ellipsis (...). No markdown formatting.

## OUTPUT (valid JSON only):
{{\n  "word_count": 165,\n  "anime_title": "Anime Name",\n  "script": "Full narration script with blank lines between paragraphs.",\n  "director_note": "Pace: Slow, dramatic.",\n  "tts_script": "[monotone] Script with tone tags for TTS.",\n  "characters": [{{"name": "Character Name", "search_query": "Character Name anime series name anime character"}}],\n  "scenes": [{{"text": "Sentence or half-sentence for 2-3s of narration", "character": "Character Name"}}]\n}}'''

script_body = {'contents': [{'role': 'user', 'parts': [{'text': PROMPT.format(topic=topic)}]}],
               'systemInstruction': {'parts': [{'text': 'Return valid JSON only. Use accurate anime knowledge. Script must be 150-180 words.'}]},
               'generationConfig': {'responseMimeType': 'application/json', 'temperature': 0.7}}

resp = requests.post(
    f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={gemini_api_key.strip()}',
    headers={'Content-Type': 'application/json'}, json=script_body, timeout=60)
resp.raise_for_status()
raw = resp.json()['candidates'][0]['content']['parts'][0]['text'].strip()
raw = re.sub(r'^```json\\s*', '', raw); raw = re.sub(r'\\s*```$', '', raw)
script_data = json.loads(raw)

script_text = script_data.get('script', '')
characters = script_data.get('characters', [])
scenes = script_data.get('scenes', [])
director_note = script_data.get('director_note', 'Pace: Slow, dramatic.')
tts_script = script_data.get('tts_script', '')
anime_title = script_data.get('anime_title', 'Unknown')

(P / 'script.txt').write_text(script_text, encoding='utf-8')
(P / 'script_data.json').write_text(json.dumps(script_data, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'   ✅ {len(script_text.split())} words | {len(characters)} characters | {len(scenes)} scenes')
print(f'   📺 Anime: {anime_title}')

# ==========================================
# STEP 2: FETCH CHARACTER IMAGES
# ==========================================
print(f'\n🖼️ [2/5] Tải ảnh nhân vật anime...')

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
TARGET_W, TARGET_H = 1080, 1920

def search_google_images(query, num=5):
    try:
        resp = requests.get('https://www.google.com/search',
            params={'q': query, 'tbm': 'isch', 'ijn': '0', 'tbs': 'isz:l'},
            headers=HEADERS, timeout=15)
        urls = []
        pattern = r'\["(https?://[^"]+\.(?:jpg|jpeg|png|webp)(?:\?[^"]*)?)",[0-9]+,[0-9]+'
        for url in re.findall(pattern, resp.text):
            if 'gstatic.com' not in url and 'google.com' not in url:
                urls.append(url)
            if len(urls) >= num: break
        if len(urls) < num:
            for url in re.findall(r'https?://[^\\"]+\.(?:jpg|jpeg|png|webp)', resp.text):
                if url not in urls and 'gstatic' not in url and 'google' not in url and 'favicon' not in url:
                    urls.append(url)
                if len(urls) >= num: break
        return urls[:num]
    except: return []

def resize_crop_vertical(img_path, out_path):
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    ratio = TARGET_W / TARGET_H
    if w/h > ratio:
        new_h, new_w = TARGET_H, int(w * (TARGET_H / h))
    else:
        new_w, new_h = TARGET_W, int(h * (TARGET_W / w))
    img = img.resize((new_w, new_h), Image.LANCZOS)
    left = (new_w - TARGET_W) // 2
    top = (new_h - TARGET_H) // 2
    img.crop((left, top, left + TARGET_W, top + TARGET_H)).save(out_path, 'JPEG', quality=92)

char_images = {}
for char in characters:
    name = char['name']
    query = char.get('search_query', f'{name} anime character')
    slug_c = re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')
    final = P / 'images' / f'{slug_c}.jpg'
    if final.exists() and final.stat().st_size > 10000:
        char_images[name] = final; print(f'   ✅ Đã có: {slug_c}.jpg'); continue
    print(f'   🔍 Tìm: "{query}"...')
    urls = search_google_images(query)
    for url in urls:
        try:
            tmp = P / 'images' / f'_tmp_{slug_c}.jpg'
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code == 200 and len(r.content) > 5000:
                tmp.write_bytes(r.content)
                resize_crop_vertical(tmp, final)
                tmp.unlink(missing_ok=True)
                char_images[name] = final
                print(f'      ✅ {slug_c}.jpg'); break
        except: continue
    time.sleep(1)

# Map scenes to images
fallback_img = list(char_images.values())[0] if char_images else None
scene_images = []
for i, sc in enumerate(scenes):
    img = char_images.get(sc.get('character', ''), fallback_img)
    if img and img.exists():
        scene_images.append(img)
    elif fallback_img:
        scene_images.append(fallback_img)
    else:
        ph = P / 'images' / f'placeholder_{i:02d}.jpg'
        Image.new('RGB', (TARGET_W, TARGET_H), (20, 20, 30)).save(ph, 'JPEG')
        scene_images.append(ph)
print(f'   ✅ {len(scene_images)} scenes mapped')

# ==========================================
# STEP 3: GENERATE TTS AUDIO
# ==========================================
print(f'\n🎤 [3/5] Tạo giọng đọc (Gemini TTS - {voice_name})...')

def convert_to_wav(audio_data, mime_type):
    bits, rate = 16, 24000
    for p in mime_type.split(';'):
        p = p.strip()
        if p.lower().startswith('rate='): rate = int(p.split('=',1)[1])
        elif p.startswith('audio/L'): bits = int(p.split('L',1)[1])
    bps = bits // 8; ba = bps; br = rate * ba
    header = struct.pack('<4sI4s4sIHHIIHH4sI', b'RIFF', 36+len(audio_data), b'WAVE',
        b'fmt ', 16, 1, 1, rate, br, ba, bits, b'data', len(audio_data))
    return header + audio_data

tts_prompt = f"""Read the following transcript based on the director's note.\n\n# Director's note\n{director_note}\n\n## Transcript:\n{tts_script if tts_script else script_text}"""

client = genai.Client(api_key=gemini_api_key)
tts_contents = [types.Content(role='user', parts=[types.Part.from_text(text=tts_prompt)])]
tts_config = types.GenerateContentConfig(
    temperature=1, response_modalities=['audio'],
    speech_config=types.SpeechConfig(
        voice_config=types.VoiceConfig(
            prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name=voice_name))))

all_audio = b''
final_mime = None
for chunk in client.models.generate_content_stream(
    model='gemini-3.1-flash-tts-preview', contents=tts_contents, config=tts_config):
    if chunk.parts is None: continue
    if chunk.parts[0].inline_data and chunk.parts[0].inline_data.data:
        all_audio += chunk.parts[0].inline_data.data
        if final_mime is None: final_mime = chunk.parts[0].inline_data.mime_type

assert all_audio, '❌ No audio data received!'
ext = mimetypes.guess_extension(final_mime) if final_mime else None
if ext is None:
    ext = '.wav'
    all_audio = convert_to_wav(all_audio, final_mime or 'audio/L16;rate=24000')
audio_path = P / f'audio{ext}'
audio_path.write_bytes(all_audio)
print(f'   ✅ Audio: {audio_path.name}')
display(Audio(str(audio_path), autoplay=False))

# ==========================================
# STEP 4: GENERATE SUBTITLES
# ==========================================
print(f'\n📝 [4/5] Tạo phụ đề (.ass)...')

# Get audio duration
try:
    result = subprocess.run(['ffmpeg', '-i', str(audio_path)], capture_output=True, text=True, errors='ignore')
    match = re.search(r'Duration:\s*(\d+):(\d+):(\d+\.\d+)', result.stderr)
    audio_dur = int(match.group(1))*3600 + int(match.group(2))*60 + float(match.group(3)) if match else 0
except: audio_dur = 0
if audio_dur <= 0: audio_dur = len(script_text.split()) / 3.0
print(f'   🕐 Duration: {audio_dur:.1f}s')

# Calculate timings
n_scenes = len(scenes)
dur_per = audio_dur / n_scenes if n_scenes else 3.0
dur_per = max(2.0, min(4.0, dur_per))
total_calc = dur_per * n_scenes
if total_calc > audio_dur: dur_per = audio_dur / n_scenes
scene_timings = []
cur = 0.0
for i in range(n_scenes):
    end = audio_dur if i == n_scenes - 1 else cur + dur_per
    scene_timings.append((cur, end))
    cur = end

# Generate .ass
def fmt_t(s):
    h=int(s//3600); m=int((s%3600)//60); sec=int(s%60); ms=int((s%1)*100)
    return f'{h}:{m:02d}:{sec:02d}.{ms:02d}'

ass_header = """[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920
WrapStyle: 1

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,62,&H00FFFFFF,&H000000FF,&H00000000,&H96000000,-1,0,0,0,100,100,0,0,3,8,0,2,50,50,130,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
all_chars = set(c['name'] for c in characters)
events = []
for sc, timing in zip(scenes, scene_timings):
    text = sc['text'].strip()
    for cn in all_chars:
        for n in [cn] + ([cn.split()[0]] if ' ' in cn else []):
            text = re.sub(r'\b'+re.escape(n)+r'\b',
                lambda m: f'{{\\c&H00FFFF&\\b1}}{m.group(0)}{{\\c&HFFFFFF&\\b0}}', text, flags=re.I)
    events.append(f'Dialogue: 0,{fmt_t(timing[0])},{fmt_t(timing[1])},Default,,0,0,0,,{text}')
sub_path = P / 'subtitles.ass'
sub_path.write_text(ass_header + '\n'.join(events), encoding='utf-8')
print(f'   ✅ subtitles.ass')

# ==========================================
# STEP 5: RENDER VIDEO
# ==========================================
print(f'\n🎬 [5/5] Render video...')

clips = []
for i, (img, timing) in enumerate(zip(scene_images, scene_timings)):
    dur = timing[1] - timing[0]
    if dur <= 0: dur = 2.5
    df = max(int(dur * 30), 1)
    clip = P / f'_clip_{i:03d}.mp4'
    clips.append(clip)
    zf = f"zoompan=z='min(zoom+0.0003,1.08)':x='iw/2-(iw/zoom)/2':y='ih/2-(ih/zoom)/2':d={df}:s=1080x1920:fps=30"
    subprocess.run(['ffmpeg', '-y', '-loop', '1', '-i', str(img), '-vf', zf,
        '-c:v', 'libx264', '-t', f'{dur:.3f}', '-pix_fmt', 'yuv420p', '-r', '30', str(clip)],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f'\r      📹 Clip {i+1}/{n_scenes}', end='', flush=True)
print()

# Concat
cl = P / '_concat.txt'
cl.write_text('\n'.join(f"file '{c.name}'" for c in clips))
merged = P / '_merged.mp4'
subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(cl),
    '-c', 'copy', str(merged)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Final merge with audio + subtitles
final_output = P / 'final_short.mp4'
ass_esc = str(sub_path).replace('\\', '/').replace(':', '\\:')
r = subprocess.run(['ffmpeg', '-y', '-i', str(merged), '-i', str(audio_path),
    '-vf', f"ass='{ass_esc}'", '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
    '-c:a', 'aac', '-b:a', '192k', '-shortest', str(final_output)],
    capture_output=True, text=True, errors='ignore')
if r.returncode != 0:
    subprocess.run(['ffmpeg', '-y', '-i', str(merged), '-i', str(audio_path),
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
        '-c:a', 'aac', '-b:a', '192k', '-shortest', str(final_output)],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Cleanup
for c in clips:
    if c.exists(): c.unlink()
if cl.exists(): cl.unlink()
if merged.exists(): merged.unlink()

size_mb = final_output.stat().st_size / (1024*1024)
print(f"\n{'='*60}")
print(f'🎉 VIDEO HOÀN TẤT! ({size_mb:.1f} MB)')
print(f'   {final_output}')
print(f"{'='*60}")

In [ ]:
# @title 📥 CELL 4: TẢI VIDEO VỀ MÁY
from google.colab import files
files.download(str(final_output))
print('Đang tải video...')